# 数据库与增删改查

学习目标：用 SQLModel 和 SQLite 保存学习记录，实现增删改查，区分提交、回滚与关闭，并验证已提交数据在服务重启后仍可读取。

前置知识：SQL 基本操作与事务、HTTP 方法、Pydantic 模型、依赖注入与上下文管理器。

适用版本：Python 3.12、SQLModel 0.0.42、SQLAlchemy 2.0；完整依赖版本见环境入口。

环境准备：[FastAPI 环境与运行入口](README.md)。工作目录为 content/Web与应用开发/FastAPI；从空内核按顺序执行。数据库位于本章创建的临时目录，末节需要独立终端启动、停止和重启端口 8100 的服务，随后执行清理。中途停止时也应先关闭服务，再释放 Engine 和临时目录。

配套脚本：位于 scripts/10-database-and-crud/。

（1）[app.py](scripts/10-database-and-crud/app.py)：只提供重启实验所需的创建和读取接口；本章 CRUD 讲解与观察直接在 Notebook 中完成。

## 1 定义一张学习记录表

SQLModel 结合 Pydantic 的数据模型与 SQLAlchemy 的数据库操作。继承 SQLModel 并设置 table=True，表示这个类映射一张数据库表；普通输入或输出模型不需要 table=True。

一条记录包含主键 id、标题 title 和允许为空的备注 note。id 的 None 默认值用于表示尚未由数据库分配编号；primary_key=True 声明主键。这里显式把表命名为 learning_record，让 Notebook 与重启实验使用同一张表。

In [1]:
from sqlmodel import Field, Session, SQLModel, create_engine, select


class Record(SQLModel, table=True):
    __tablename__ = "learning_record"

    id: int | None = Field(default=None, primary_key=True)
    title: str
    note: str | None = None


draft_record = Record(title="阅读数据库文档", note="先理解事务")
print(draft_record.model_dump())
# 创建 Python 对象还没有插入数据库，也没有自动生成主键。
assert draft_record.id is None

{'title': '阅读数据库文档', 'note': '先理解事务', 'id': None}


## 2 创建 Engine 和数据库文件

Engine 管理数据库连接的获取与连接池；Session 管理一次工作中的对象与事务。下面使用临时目录中的文件数据库，避免把演示数据写进课程目录。

sqlite:/// 后接数据库文件路径。create_all 根据已声明的模型创建缺少的表；它不负责把已有表自动改成新的字段结构。为已有数据增加列、改变约束等变更，应使用数据库迁移流程；本章只初始化这张小表。

check_same_thread=False 放宽 SQLite 驱动对连接使用线程的检查，便于同步接口与依赖使用文件数据库。SQLAlchemy 2.0 的文件数据库默认也采用这一设置；它不意味着可以并发共享同一个 Session。

In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory


database_directory = TemporaryDirectory(prefix="fastapi-records-")
database_path = Path(database_directory.name) / "records.sqlite3"
engine = create_engine(
    f"sqlite:///{database_path.as_posix()}",
    connect_args={"check_same_thread": False},
)
SQLModel.metadata.create_all(engine)
print("数据库文件：", database_path.as_posix())
assert database_path.is_file()
# 必须先定义 Record，再调用 create_all；表模型会注册到 metadata。

数据库文件： C:/Users/ZHUANG/AppData/Local/Temp/fastapi-records-gszlxrvr/records.sqlite3


## 3 插入记录并提交事务

session.add 将对象加入当前工作，commit 将本次事务提交到数据库。refresh 重新读取对象状态，便于获取数据库生成的编号等内容。

with Session(engine) 在退出时关闭 Session、释放其持有的连接资源，但不会替代显式提交。需要写入的数据在代码中调用 commit；纯读取通常不需要 commit。

In [3]:
with Session(engine) as session:
    session.add(draft_record)
    session.commit()
    session.refresh(draft_record)
    first_id = draft_record.id
    print(draft_record.model_dump())
    assert isinstance(first_id, int)

# 用新的 Session 读取，确认读取的不是刚才那个 Python 对象。
with Session(engine) as session:
    stored = session.get(Record, first_id)
    assert stored is not None
    assert stored.title == "阅读数据库文档"
    print("新会话读取：", stored.title)

{'id': 1, 'title': '阅读数据库文档', 'note': '先理解事务'}
新会话读取： 阅读数据库文档


## 4 查询、排序和分页

session.get 按主键取一条记录，找不到时得到 None；select 描述查询，session.exec 执行查询。order_by 明确排序，offset 跳过指定数量的记录，limit 限制返回条数。

下面按唯一主键升序排列，再读取第二条记录。用 .desc() 可以反向排序。分页前明确顺序，才能清楚解释每一页从哪里开始；本例查询期间没有其他写入。

In [4]:
with Session(engine) as session:
    session.add(Record(title="练习查询"))
    session.add(Record(title="练习更新"))
    session.commit()

with Session(engine) as session:
    statement = select(Record).order_by(Record.id).offset(1).limit(1)
    page = session.exec(statement).all()
    newest = session.exec(select(Record).order_by(Record.id.desc())).first()
    print("第二条：", [record.title for record in page])
    print("倒序首条：", newest.title)
    assert [record.title for record in page] == ["练习查询"]
    assert newest.title == "练习更新"
    assert session.get(Record, 999) is None

第二条： ['练习查询']
倒序首条： 练习更新


## 5 失败后回滚，关闭时释放资源

主键重复会触发数据库的完整性约束。下面在同一个事务中添加两条记录，其中一条故意与已有主键冲突；整次事务都不能保留。

flush 会把待处理的变更发送给数据库，commit 会先执行所需的 flush。发生约束失败后，Session 进入需要处理的失败状态；若还要继续使用这个 Session，先显式 rollback。退出 with 则关闭它、释放连接资源，关闭本身不等于提交成功。

In [5]:
from sqlalchemy.exc import IntegrityError


with Session(engine) as session:
    session.add(Record(id=999, title="本次应该撤销"))
    session.add(Record(id=first_id, title="重复主键"))
    try:
        session.commit()
    except IntegrityError:
        session.rollback()
        print("主键冲突：本次事务已回滚")
    else:
        raise AssertionError("应当触发主键约束")

    # 回滚后，同一个 Session 可以重新查询；另一条新增也没有保留。
    assert session.get(Record, 999) is None
    original = session.get(Record, first_id)
    assert original.title == "阅读数据库文档"
    print("原记录仍为：", original.title)

主键冲突：本次事务已回滚
原记录仍为： 阅读数据库文档


## 6 分开创建模型、输出模型和请求会话

数据库模型需要描述表，创建模型描述客户端可以提交什么，输出模型描述接口承诺返回什么。创建时不接收主键；输出时 id 已经由数据库分配，因此输出模型要求整数 id。

本章使用同步数据库驱动，路由使用 def。yield 依赖为每次请求创建一个 Session，退出 with 时关闭。Engine 可以由应用共享；Session 是可变的事务状态，不应作为全局对象供并发请求或多个线程同时使用。check_same_thread=False 不改变这个边界。

In [6]:
from typing import Annotated

from fastapi import Depends, FastAPI, HTTPException, Query
from fastapi.testclient import TestClient


class RecordCreate(SQLModel):
    title: str = Field(min_length=1)
    note: str | None = None


class RecordPublic(SQLModel):
    id: int
    title: str
    note: str | None


def get_session():
    with Session(engine) as session:
        yield session


SessionDependency = Annotated[Session, Depends(get_session)]
app = FastAPI()

C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 7 创建和读取接口

POST /records 把经过输入校验的数据转换为表模型，提交后返回公开模型。GET /records/{record_id} 按整数编号读取；不存在时明确返回 404。

commit 与 refresh 在会话有效期间完成。yield 依赖负责会话退出，但业务代码仍决定什么时候提交；未处理的异常传播时，上下文也会释放当前会话。

In [7]:
@app.post("/records", response_model=RecordPublic, status_code=201)
def create_record(payload: RecordCreate, session: SessionDependency):
    record = Record.model_validate(payload)
    session.add(record)
    session.commit()
    session.refresh(record)
    return record


@app.get("/records/{record_id}", response_model=RecordPublic)
def read_record(record_id: int, session: SessionDependency):
    record = session.get(Record, record_id)
    if record is None:
        raise HTTPException(status_code=404, detail="记录不存在")
    return record

先用应用内调用创建一条记录，再用返回的编号读取。输入中空标题会被模型拒绝；业务查询不存在与请求校验失败分别观察。

In [8]:
with TestClient(app) as client:
    created = client.post(
        "/records", json={"title": "通过接口创建", "note": "保留这条备注"}
    )
    api_id = created.json()["id"]
    loaded = client.get(f"/records/{api_id}")
    missing = client.get("/records/999")
    invalid = client.post("/records", json={"title": ""})
    print(created.status_code, created.json())
    print("不存在与非法输入：", missing.status_code, invalid.status_code)
    assert created.status_code == 201
    assert loaded.status_code == 200 and loaded.json() == created.json()
    assert missing.status_code == 404
    assert invalid.status_code == 422

201 {'id': 4, 'title': '通过接口创建', 'note': '保留这条备注'}
不存在与非法输入： 404 422


## 8 让列表接口限制返回数量

列表接口接收 offset 和 limit 查询参数。offset 至少为 0，limit 在 1 到 100 之间；这些是本例设置的分页边界。查询仍按 id 升序执行，再分页。

单条读取和列表读取都是数据库查询，不需要先把整张表读入 Python 再切片。

In [9]:
@app.get("/records", response_model=list[RecordPublic])
def list_records(
    session: SessionDependency,
    offset: Annotated[int, Query(ge=0)] = 0,
    limit: Annotated[int, Query(ge=1, le=100)] = 20,
):
    statement = select(Record).order_by(Record.id).offset(offset).limit(limit)
    return session.exec(statement).all()


with TestClient(app) as client:
    page = client.get("/records", params={"offset": 1, "limit": 2})
    print(page.json())
    assert page.status_code == 200
    assert [record["title"] for record in page.json()] == ["练习查询", "练习更新"]
    assert client.get("/records", params={"limit": 101}).status_code == 422
    assert client.get("/records", params={"offset": -1}).status_code == 422

[{'id': 2, 'title': '练习查询', 'note': None}, {'id': 3, 'title': '练习更新', 'note': None}]


## 9 部分更新区分未提供和显式空值

PATCH 只修改本次提供的字段。更新模型为字段提供默认值，允许调用方省略它们；model_dump(exclude_unset=True) 只保留显式提供的字段。

省略 note 表示保留原备注，传入 note: null 表示清空备注。title 不允许在数据库中为空：本例允许更新模型用 None 表示缺省，但在接口中明确拒绝显式提交的 title: null。

In [10]:
class RecordUpdate(SQLModel):
    title: str | None = Field(default=None, min_length=1)
    note: str | None = None


omitted = RecordUpdate(title="新标题")
explicit_null = RecordUpdate(note=None)
print("未提供备注：", omitted.model_dump(exclude_unset=True))
print("显式清空备注：", explicit_null.model_dump(exclude_unset=True))
assert "note" not in omitted.model_dump(exclude_unset=True)
assert explicit_null.model_dump(exclude_unset=True) == {"note": None}

未提供备注： {'title': '新标题'}
显式清空备注： {'note': None}


sqlmodel_update 将更新字典应用到已加载的表对象，然后提交并刷新。不存在的记录直接返回 404；显式空标题在修改对象前拒绝，避免把输入问题留给数据库约束处理。

In [11]:
@app.patch("/records/{record_id}", response_model=RecordPublic)
def update_record(
    record_id: int, payload: RecordUpdate, session: SessionDependency
):
    record = session.get(Record, record_id)
    if record is None:
        raise HTTPException(status_code=404, detail="记录不存在")
    changes = payload.model_dump(exclude_unset=True)
    if "title" in changes and changes["title"] is None:
        raise HTTPException(status_code=422, detail="title 不能为 null")
    record.sqlmodel_update(changes)
    session.add(record)
    session.commit()
    session.refresh(record)
    return record

连续执行“只改标题”“清空备注”“拒绝空标题”，再从数据库重新读取，核对保存结果。每次请求使用自己的 Session，最终查询不依赖前一次请求中的对象。

In [12]:
with TestClient(app) as client:
    renamed = client.patch(f"/records/{api_id}", json={"title": "已经修改"})
    cleared = client.patch(f"/records/{api_id}", json={"note": None})
    rejected = client.patch(f"/records/{api_id}", json={"title": None})
    print(renamed.json())
    print(cleared.json())
    print(rejected.status_code, rejected.json())
    assert renamed.json()["note"] == "保留这条备注"
    assert cleared.json()["note"] is None
    assert rejected.status_code == 422
    assert client.patch("/records/999", json={"note": None}).status_code == 404

with Session(engine) as session:
    stored = session.get(Record, api_id)
    assert stored.title == "已经修改" and stored.note is None

{'id': 4, 'title': '已经修改', 'note': '保留这条备注'}
{'id': 4, 'title': '已经修改', 'note': None}
422 {'detail': 'title 不能为 null'}


## 10 删除需要提交，也需要处理不存在

session.delete 标记要删除的对象，commit 执行并提交删除。本例删除成功返回 {"ok": true}，重复删除同一编号返回 404。

再次查询数据库，确认记录已经删除，而不仅是收到了一条成功消息。

In [13]:
@app.delete("/records/{record_id}")
def delete_record(record_id: int, session: SessionDependency):
    record = session.get(Record, record_id)
    if record is None:
        raise HTTPException(status_code=404, detail="记录不存在")
    session.delete(record)
    session.commit()
    return {"ok": True}


with TestClient(app) as client:
    deleted = client.delete(f"/records/{api_id}")
    repeated = client.delete(f"/records/{api_id}")
    print(deleted.status_code, deleted.json(), repeated.status_code)
    assert deleted.status_code == 200 and deleted.json() == {"ok": True}
    assert repeated.status_code == 404

with Session(engine) as session:
    assert session.get(Record, api_id) is None

200

 {'ok': True} 404


## 11 重启真实服务后读取已提交数据

文件数据库可以由新进程重新打开。配套 app.py 只保留本章相同表结构与创建、读取路由；create_app 在启动时读取 FASTAPI_DATABASE_PATH，lifespan 初始化表并在退出时 dispose Engine。

这里使用同一个数据库文件，先结束 Notebook 当前连接池的使用。dispose 释放池中空闲的连接；已有 Session 应先关闭，不能用 dispose 代替关闭仍在使用的会话。

Step 1：执行下面单元，再把输出的赋值命令复制到已激活课程环境、位于课程目录的独立 PowerShell 终端。

In [14]:
engine.dispose()
print(f"$env:FASTAPI_DATABASE_PATH = '{database_path.as_posix()}'")
# 数据库文件仍然存在；dispose 不会删除已提交的数据。
assert database_path.is_file()

$env:FASTAPI_DATABASE_PATH = 'C:/Users/ZHUANG/AppData/Local/Temp/fastapi-records-gszlxrvr/records.sqlite3'


Step 2：在该终端启动服务，等待 Application startup complete，再执行下面的 HTTPX 单元。

```powershell
python -m uvicorn app:create_app --factory --app-dir scripts/10-database-and-crud --host 127.0.0.1 --port 8100
```

--factory 表示调用 create_app 得到应用；导入模块不会启动服务器。HTTPX 这次访问真实端口，timeout 限制网络等待，trust_env=False 避免本地调用受到代理环境变量影响。

In [15]:
import httpx


with httpx.Client(
    base_url="http://127.0.0.1:8100", timeout=5, trust_env=False
) as client:
    service_created = client.post(
        "/records", json={"title": "重启后继续阅读", "note": "已提交"}
    )
    assert service_created.status_code == 201
    persisted_record = service_created.json()
    print("第一次启动写入：", persisted_record)
# 关闭 HTTPX 客户端不会关闭 Uvicorn，也不会删除数据库。

第一次启动写入： {'id': 4, 'title': '重启后继续阅读', 'note': '已提交'}


Step 3：在服务终端按 Ctrl+C，等待退出并回到命令提示符；保留数据库文件和环境变量。

Step 4：再次运行同一启动命令，等待 Application startup complete，然后执行读取单元。

```powershell
python -m uvicorn app:create_app --factory --app-dir scripts/10-database-and-crud --host 127.0.0.1 --port 8100
```

In [16]:
with httpx.Client(
    base_url="http://127.0.0.1:8100", timeout=5, trust_env=False
) as client:
    loaded_after_restart = client.get(f"/records/{persisted_record['id']}")
    print("重启后读取：", loaded_after_restart.status_code, loaded_after_restart.json())
    assert loaded_after_restart.status_code == 200
    assert loaded_after_restart.json() == persisted_record
# 新进程从同一个 SQLite 文件读到已经提交的记录。

重启后读取： 200 {'id': 4, 'title': '重启后继续阅读', 'note': '已提交'}


Step 5：再次在服务终端按 Ctrl+C，等待服务完全退出，然后运行下面单元清理本章临时数据库。

In [17]:
engine.dispose()
database_directory.cleanup()
assert not database_path.exists()
print("本章 Engine 已释放，临时数据库目录已清理")
# 若中途停止实验，也应先关闭服务，再执行这个清理单元。

本章 Engine 已释放，临时数据库目录已清理


Step 6：关闭用于本实验的独立终端，结束该终端中指向临时数据库的 FASTAPI_DATABASE_PATH 设置。

## 本章小结

（1）表模型负责映射数据库，创建模型和输出模型负责接口输入与输出；主键由数据库分配后再返回。

（2）commit 保存事务，rollback 撤销当前事务；关闭 Session 释放连接资源，不会替代提交。约束失败后再次使用同一会话前需要回滚。

（3）请求各自创建会话，不并发共享 Session。排序、分页和不存在响应都在查询接口中明确表达。

（4）部分更新只应用显式提供的字段：省略保持原值，允许为空的字段传 null 则清空。重启后能否读取，取决于写入是否提交、是否重新打开同一文件。

## 练习

重新运行到对应示例，在末尾清理前完成练习；完成后关闭服务并清理本章临时数据库。

（1）把列表接口改为按 id 降序，读取 limit=2 的第一页与 offset=2 的第二页。确认两个页面的编号顺序符合查询规则，且本次静态数据中不重复。

（2）对同一条记录依次提交空更新对象、只修改 title、note: null。每次都从新的 Session 读取，确认分别保留所有值、只改标题、清空备注。

（3）在一个事务中先添加普通记录，再添加冲突主键。核对 rollback 后两条操作都未改变已提交结果，并确认同一 Session 可以再次执行查询。

（4）在真实服务中创建两条不同标题的记录，关闭并重启服务，再逐条读取。确认编号与内容一致，最后关闭服务并清理临时目录。

### 提示

（1）先确定 order_by，再应用 offset 和 limit；修改后的断言也应使用新的排序结果。

（2）观察 model_dump(exclude_unset=True) 的字典，而不是仅比较字段的默认值。

（3）捕获具体的 IntegrityError；出现其他异常应继续报告，不要把所有失败都当作主键冲突。

（4）重启前不要删除数据库，也不要更换 FASTAPI_DATABASE_PATH；清理必须在服务退出之后。

## 参考与引用来源

- SQLModel 官方文档：[Create a Table](https://sqlmodel.tiangolo.com/tutorial/create-db-and-table/) 的 table=True、Engine、metadata 与 Migrations；[Create Rows](https://sqlmodel.tiangolo.com/tutorial/insert/) 与 [Automatic IDs and Refresh](https://sqlmodel.tiangolo.com/tutorial/automatic-id-none-refresh/) 的 add、commit、refresh；[Read One Row](https://sqlmodel.tiangolo.com/tutorial/one/) 与 [Limit and Offset](https://sqlmodel.tiangolo.com/tutorial/limit-and-offset/)；[Multiple Models](https://sqlmodel.tiangolo.com/tutorial/fastapi/multiple-models/)、[Session Dependency](https://sqlmodel.tiangolo.com/tutorial/fastapi/session-with-dependency/)、[Read One Model](https://sqlmodel.tiangolo.com/tutorial/fastapi/read-one/)；[Update Data](https://sqlmodel.tiangolo.com/tutorial/fastapi/update/) 的 exclude_unset、sqlmodel_update、显式 None；[Delete Data](https://sqlmodel.tiangolo.com/tutorial/fastapi/delete/)。支持表定义、事务操作、输入输出模型和 CRUD 接口。
- SQLAlchemy 2.0 官方文档：[Session Basics](https://docs.sqlalchemy.org/en/20/orm/session_basics.html) 的 Committing、Rolling Back、Closing、Is the Session thread-safe；[SQLite](https://docs.sqlalchemy.org/en/20/dialects/sqlite.html#threading-pooling-behavior) 的文件数据库连接池与 check_same_thread；[ORDER BY](https://docs.sqlalchemy.org/en/20/tutorial/data_select.html#order-by)；[MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#creating-and-dropping-database-tables) 的 create_all 与迁移边界；[Engine Disposal](https://docs.sqlalchemy.org/en/20/core/connections.html#engine-disposal)。支持失败回滚、会话并发范围、排序及连接释放。
- FastAPI 官方文档：[SQL Databases](https://fastapi.tiangolo.com/tutorial/sql-databases/) 的 SessionDependency 与同步接口；[Query Parameters and Numeric Validations](https://fastapi.tiangolo.com/tutorial/path-params-numeric-validations/) 的 Query 数值边界；[Lifespan](https://fastapi.tiangolo.com/advanced/events/) 的应用资源初始化和退出；[Testing](https://fastapi.tiangolo.com/tutorial/testing/) 的 TestClient。
- Uvicorn 官方文档：[Settings](https://uvicorn.dev/settings/) 的 Application、--factory、--app-dir 与 Socket Binding，支持独立进程的启动命令。
- HTTPX 官方文档：[QuickStart](https://www.python-httpx.org/quickstart/) 的客户端请求与 JSON；[Timeouts](https://www.python-httpx.org/advanced/timeouts/) 与 [Environment Variables](https://www.python-httpx.org/environment_variables/) 的 timeout、trust_env。
- Python 3.12 官方文档：[TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory) 的临时目录清理；[pathlib](https://docs.python.org/3.12/library/pathlib.html#pathlib.PurePath.as_posix) 的路径表示，支持本章临时数据库路径与清理操作。